# 🔥 PyTorch — Complete Revision Guide

> **PyTorch** is developed by Meta AI. Unlike TensorFlow 1.x, it uses **dynamic computation graphs** — the graph is built as code runs (define-by-run), making debugging natural and Pythonic.

---

## 📚 What You Will Learn
1. Setup & Tensors  
2. Autograd — How gradients work  
3. nn.Module — Building models  
4. Loss Functions & Optimizers  
5. Training Loop from scratch  
6. Dataset & DataLoader  
7. Full MNIST Example (MLP + CNN)  
8. LSTM Sequence Model  
9. Transfer Learning  
10. Saving & Loading  
11. GPU support  
12. Interview Q&A  

---
## 1️⃣ Installation & Verification

In [ ]:
# Install PyTorch (run in terminal if not installed):
# pip install torch torchvision

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

# Use GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {device}')

---
## 2️⃣ Tensors — The Core Building Block

A tensor is an N-dimensional array. Think of it as NumPy array + GPU support + autograd.

In [ ]:
import torch
import numpy as np

print('=== Creating Tensors ===')
t1 = torch.tensor([1.0, 2.0, 3.0])           # From list
t2 = torch.zeros(3, 4)                        # All zeros
t3 = torch.ones(2, 3)                         # All ones
t4 = torch.rand(3, 3)                         # Uniform [0, 1)
t5 = torch.randn(3, 3)                        # Standard normal N(0,1)
t6 = torch.arange(0, 10, 2)                   # [0, 2, 4, 6, 8]
t7 = torch.linspace(0, 1, 5)                  # 5 evenly spaced values
t8 = torch.eye(3)                             # Identity matrix

print(f'tensor([1,2,3])  : {t1}')
print(f'zeros(3,4) shape : {t2.shape}')
print(f'rand(3,3):\n{t4.round(decimals=3)}')
print(f'arange(0,10,2)   : {t6}')
print(f'linspace(0,1,5)  : {t7}')

print('\n=== NumPy ↔ PyTorch ===')
arr = np.array([1.0, 2.0, 3.0])
from_np  = torch.from_numpy(arr)   # Shares memory! Changes reflect in both
from_np2 = torch.tensor(arr)       # Copies data
back_np  = from_np.numpy()         # Back to numpy

arr[0] = 99
print(f'After arr[0]=99, from_numpy: {from_np[0]}')  # 99 — shared memory!
print(f'After arr[0]=99, tensor()  : {from_np2[0]}')  # 1.0 — copied

In [ ]:
import torch

print('=== Tensor Operations ===')
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print('Element-wise add  :', (a + b).tolist())
print('Element-wise mul  :', (a * b).tolist())
print('Matrix multiply   :\n', torch.mm(a, b).tolist())
print('a @ b (same)      :\n', (a @ b).tolist())

print('\n=== Shape Operations ===')
t = torch.randn(2, 3, 4)
print(f'Original          : {t.shape}')            # [2, 3, 4]
print(f'view(2, 12)       : {t.view(2, 12).shape}')# [2, 12]
print(f'reshape(6, 4)     : {t.reshape(6, 4).shape}')# [6, 4]
print(f'unsqueeze(0)      : {t.unsqueeze(0).shape}') # [1, 2, 3, 4]
print(f'squeeze all dims  : {t.squeeze().shape}')    # removes size-1 dims
print(f'permute(2,0,1)    : {t.permute(2,0,1).shape}')# [4, 2, 3]

print('\n=== Reduction Operations ===')
x = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print(f'sum()             : {x.sum()}')
print(f'sum(axis=0)       : {x.sum(dim=0)}')  # column sums
print(f'mean()            : {x.mean()}')
print(f'max()             : {x.max()}')
print(f'argmax(dim=1)     : {x.argmax(dim=1)}')  # index of max per row

print('\n=== Type Casting ===')
t_int = torch.tensor([1, 2, 3])
print(f'int tensor dtype  : {t_int.dtype}')
print(f'after .float()    : {t_int.float().dtype}')
print(f'after .to(float64): {t_int.to(torch.float64).dtype}')

---
## 3️⃣ Autograd — How PyTorch Computes Gradients

Every operation on a `requires_grad=True` tensor is tracked in a **computation graph**.  
Calling `.backward()` walks the graph backwards to compute derivatives.

In [ ]:
import torch

# Simple scalar example: y = x^2 + 2x + 1
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1

print(f'x = {x.item()}')
print(f'y = x^2 + 2x + 1 = {y.item()}')

y.backward()  # Computes dy/dx = 2x + 2
print(f'dy/dx = 2*{x.item()} + 2 = {x.grad.item()}')  # 8.0

print('\n--- Gradient w.r.t. multiple params ---')
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x_val = torch.tensor(3.0)

y = w * x_val + b   # y = 2*3 + 1 = 7
y.backward()
print(f'y = w*x + b = {y.item()}')
print(f'dy/dw = x = {w.grad.item()}')   # 3.0
print(f'dy/db = 1 = {b.grad.item()}')   # 1.0

print('\n--- Disable gradient tracking ---')
x2 = torch.tensor(5.0, requires_grad=True)
with torch.no_grad():           # Context manager — fastest option
    z = x2 ** 2
print(f'z.requires_grad: {z.requires_grad}')  # False

z2 = x2.detach() ** 2          # Detach from graph
print(f'z2.requires_grad: {z2.requires_grad}')  # False

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Visualize gradient descent on y = x^2 (minimum at x=0)
x = torch.tensor(8.0, requires_grad=True)
lr = 0.1
xs, ys = [x.item()], [x.item()**2]

for _ in range(30):
    if x.grad is not None:
        x.grad.zero_()
    y = x**2
    y.backward()
    with torch.no_grad():
        x -= lr * x.grad
    xs.append(x.item())
    ys.append(x.item()**2)

# Plot
xrange = np.linspace(-9, 9, 200)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(xrange, xrange**2, 'b-', linewidth=2, label='y = x²')
plt.scatter(xs, ys, c=range(len(xs)), cmap='Reds', zorder=5, s=30)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Gradient Descent on y = x²')
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(ys, 'r-o', markersize=3)
plt.xlabel('Step'); plt.ylabel('Loss (y = x²)')
plt.title('Loss Decreasing over Steps')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Final x = {x.item():.6f} (should be ~0)')

---
## 4️⃣ Building Models with nn.Module

Every PyTorch model is a class that inherits from `nn.Module`. You define:
- `__init__`: layers/components
- `forward`: how data flows through

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    """Multi-Layer Perceptron"""
    def __init__(self, input_size=784, hidden=256, num_classes=10, dropout=0.3):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes)
            # NOTE: no softmax here! CrossEntropyLoss includes it
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params   : {total:,}')
print(f'Trainable params: {trainable:,}')

# Test forward pass
dummy_input = torch.randn(4, 784)      # batch=4
out = model(dummy_input)
print(f'\nOutput shape   : {out.shape}') # (4, 10)

---
## 5️⃣ Loss Functions & Optimizers Demo

In [ ]:
import torch
import torch.nn as nn

print('=== Loss Functions ===')

# --- Classification: CrossEntropyLoss ---
# Input: raw logits (no softmax needed), Target: integer class labels
logits = torch.tensor([[2.0, 0.5, 0.1],   # Likely class 0
                        [0.1, 3.0, 0.2]])  # Likely class 1
targets = torch.tensor([0, 1])             # True classes

ce_loss = nn.CrossEntropyLoss()(logits, targets)
print(f'CrossEntropyLoss (correct preds): {ce_loss.item():.4f}')  # Low

wrong_targets = torch.tensor([2, 0])       # Wrong classes
ce_loss_wrong = nn.CrossEntropyLoss()(logits, wrong_targets)
print(f'CrossEntropyLoss (wrong preds)  : {ce_loss_wrong.item():.4f}')  # High

# --- Binary Classification ---
pred_logits = torch.tensor([2.0, -1.0, 0.5])   # raw logits
true_labels = torch.tensor([1.0, 0.0, 1.0])
bce = nn.BCEWithLogitsLoss()(pred_logits, true_labels)
print(f'BCEWithLogitsLoss               : {bce.item():.4f}')

# --- Regression ---
pred_vals  = torch.tensor([2.5, 3.0, 4.2])
true_vals  = torch.tensor([3.0, 3.0, 4.0])
mse = nn.MSELoss()(pred_vals, true_vals)
mae = nn.L1Loss()(pred_vals, true_vals)
print(f'MSE Loss                        : {mse.item():.4f}')
print(f'MAE Loss                        : {mae.item():.4f}')

print('\n=== Optimizers ===')
model_demo = nn.Linear(10, 5)

optimizers = {
    'SGD (lr=0.1)':  optim.SGD(model_demo.parameters(), lr=0.1),
    'SGD+Momentum':  optim.SGD(model_demo.parameters(), lr=0.01, momentum=0.9),
    'Adam':          optim.Adam(model_demo.parameters(), lr=0.001),
    'AdamW':         optim.AdamW(model_demo.parameters(), lr=0.001, weight_decay=0.01),
    'RMSprop':       optim.RMSprop(model_demo.parameters(), lr=0.001),
}
for name, opt in optimizers.items():
    print(f'  {name}')

---
## 6️⃣ Full Training Example — MNIST with PyTorch

We'll use `torchvision` to download MNIST and build a proper training loop.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ── Transforms: convert PIL image to tensor and normalize ──
transform = transforms.Compose([
    transforms.ToTensor(),                   # PIL → [0,1] tensor
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

# ── Download datasets ──────────────────────────────────────
train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# ── DataLoaders ─────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples : {len(test_dataset)}')
print(f'Train batches: {len(train_loader)}')

# ── Inspect one batch ────────────────────────────────────────
images, labels = next(iter(train_loader))
print(f'Batch images shape: {images.shape}')  # (128, 1, 28, 28)
print(f'Batch labels shape: {labels.shape}')  # (128,)
print(f'Pixel range after normalize: [{images.min():.2f}, {images.max():.2f}]')

# ── Visualize ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(16, 3))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(labels[i].item(), fontsize=9)
    ax.axis('off')
plt.suptitle('MNIST Samples (after normalization)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Model ───────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.net(x)

model    = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)   # Halve LR every 5 epochs

# ── Training function ────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()  # IMPORTANT: enables Dropout and BN training mode
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()              # 1. Clear old gradients
        outputs = model(images)            # 2. Forward pass
        loss    = criterion(outputs, labels)# 3. Compute loss
        loss.backward()                    # 4. Backpropagation
        optimizer.step()                   # 5. Update weights

        total_loss += loss.item() * labels.size(0)
        _, predicted = outputs.max(1)      # Argmax
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / total, correct / total

# ── Evaluation function ──────────────────────────────────────
def evaluate(model, loader, criterion, device):
    model.eval()   # IMPORTANT: disables Dropout, uses BN running stats
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():  # No gradient computation during eval
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return total_loss / total, correct / total

# ── Main training loop ───────────────────────────────────────
train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, 11):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    train_losses.append(tr_loss); val_losses.append(va_loss)
    train_accs.append(tr_acc);    val_accs.append(va_acc)

    print(f'Epoch {epoch:2d}/10 | LR:{scheduler.get_last_lr()[0]:.5f} | '
          f'Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f} | '
          f'Val Loss:{va_loss:.4f} Acc:{va_acc:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, 'b-o', label='Train Loss', markersize=4)
ax1.plot(val_losses,   'r-o', label='Val Loss',   markersize=4)
ax1.set_title('Loss Curves'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in train_accs], 'b-o', label='Train Acc', markersize=4)
ax2.plot([a*100 for a in val_accs],   'r-o', label='Val Acc',   markersize=4)
ax2.set_title('Accuracy Curves'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final Test Accuracy: {val_accs[-1]*100:.2f}%')

---
## 7️⃣ CNN with PyTorch

In [ ]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        # Feature extractor: (batch, 1, 28, 28) → (batch, 128, 7, 7)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # → (batch, 32, 28, 28)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # → (batch, 32, 14, 14)

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # → (batch, 64, 14, 14)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # → (batch, 64, 7, 7)

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),# → (batch, 128, 7, 7)
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # → (batch, 128, 1, 1) — works for any input size!
            nn.Flatten(),                  # → (batch, 128)
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Test
cnn_model = CNN(num_classes=10).to(device)
dummy = torch.randn(4, 1, 28, 28).to(device)
out   = cnn_model(dummy)
print(f'CNN output shape: {out.shape}')   # (4, 10)
total = sum(p.numel() for p in cnn_model.parameters())
print(f'CNN params      : {total:,}')

# Train
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
for epoch in range(1, 6):
    tr_loss, tr_acc = train_epoch(cnn_model, train_loader, criterion, cnn_optimizer, device)
    va_loss, va_acc = evaluate(cnn_model, test_loader, criterion, device)
    print(f'Epoch {epoch}/5 | Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f}')

print(f'\nCNN Final Test Acc: {va_acc*100:.2f}%')

---
## 8️⃣ LSTM with PyTorch

Treating MNIST images as sequences (28 timesteps × 28 features).

In [ ]:
import torch
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,    # input: (batch, seq, features)
            dropout     = dropout,
            bidirectional = False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, 28, 28) — 28 timesteps, each 28 features
        out, (h_n, c_n) = self.lstm(x)
        # out  : (batch, 28, hidden_size) — output at each timestep
        # h_n  : (num_layers, batch, hidden_size) — last hidden state
        # c_n  : (num_layers, batch, hidden_size) — last cell state

        last_out = out[:, -1, :]       # Take output of LAST timestep
        out_drop = self.dropout(last_out)
        logits   = self.fc(out_drop)
        return logits

lstm_model = LSTMClassifier().to(device)

# The input to LSTM should be (batch, seq_len, features)
# MNIST image (1, 28, 28) → squeeze → (28, 28)
def prepare_lstm_batch(images):
    return images.squeeze(1)  # (batch, 1, 28, 28) → (batch, 28, 28)

lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

for epoch in range(1, 6):
    lstm_model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in train_loader:
        images = prepare_lstm_batch(images).to(device)
        labels = labels.to(device)
        lstm_optimizer.zero_grad()
        outputs = lstm_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=1.0)  # Prevent exploding gradients!
        lstm_optimizer.step()
        total_loss += loss.item(); _, pred = outputs.max(1)
        total += labels.size(0); correct += pred.eq(labels).sum().item()

    lstm_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = prepare_lstm_batch(images).to(device)
            labels = labels.to(device)
            _, pred = lstm_model(images).max(1)
            val_total += labels.size(0)
            val_correct += pred.eq(labels).sum().item()

    print(f'Epoch {epoch}/5 | Train Acc: {correct/total:.4f} | Val Acc: {val_correct/val_total:.4f}')

---
## 9️⃣ Custom Dataset

When you have your own data, inherit from `torch.utils.data.Dataset`.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ── Custom Dataset for tabular data ──────────────────────
class TabularDataset(Dataset):
    """Custom dataset for numpy/pandas data"""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Convert to float32 tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)          # Required

    def __getitem__(self, idx):     # Required — returns ONE sample
        return self.X[idx], self.y[idx]

# Generate dummy data
np.random.seed(42)
X_fake = np.random.randn(1000, 20).astype(np.float32)
y_fake = np.random.randint(0, 3, 1000)                 # 3 classes

dataset = TabularDataset(X_fake, y_fake)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

print(f'Dataset size    : {len(dataset)}')
print(f'Number of batches: {len(loader)}')

# Inspect one item
x_sample, y_sample = dataset[0]
print(f'One sample X    : {x_sample.shape}')    # (20,)
print(f'One sample y    : {y_sample}')

# Inspect one batch
x_batch, y_batch = next(iter(loader))
print(f'Batch X shape   : {x_batch.shape}')    # (64, 20)
print(f'Batch y shape   : {y_batch.shape}')    # (64,)

# ── Train a quick model on this custom dataset ────────────
model_tab = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 3)
)
opt_tab = optim.Adam(model_tab.parameters())

for epoch in range(5):
    model_tab.train()
    for xb, yb in loader:
        opt_tab.zero_grad()
        loss = criterion(model_tab(xb), yb)
        loss.backward(); opt_tab.step()
print(f'Final batch loss: {loss.item():.4f}')

---
## 🔟 Saving & Loading Models

In [ ]:
import torch
import os

# ── Method 1: Save state_dict (RECOMMENDED) ──────────────
torch.save(cnn_model.state_dict(), '/tmp/cnn_state.pth')
print('Saved state_dict to /tmp/cnn_state.pth')

# Load — must recreate model architecture first
loaded_cnn = CNN(num_classes=10).to(device)
loaded_cnn.load_state_dict(torch.load('/tmp/cnn_state.pth', map_location=device))
loaded_cnn.eval()
print('Loaded state_dict successfully')

# ── Method 2: Save full model (NOT recommended — brittle) ──
torch.save(cnn_model, '/tmp/cnn_full.pth')
full_model = torch.load('/tmp/cnn_full.pth', map_location=device)
print('Full model save/load done')

# ── Method 3: Save checkpoint (training resume) ──────────
checkpoint = {
    'epoch'           : 5,
    'model_state_dict': cnn_model.state_dict(),
    'optimizer_state_dict': cnn_optimizer.state_dict(),
    'val_accuracy'    : va_acc,
}
torch.save(checkpoint, '/tmp/checkpoint.pth')

# Resume training from checkpoint
ckpt      = torch.load('/tmp/checkpoint.pth', map_location=device)
resume_model = CNN().to(device)
resume_model.load_state_dict(ckpt['model_state_dict'])
resume_optimizer = optim.Adam(resume_model.parameters())
resume_optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch'] + 1

print(f'\nResuming from epoch {start_epoch}')
print(f'Previous val accuracy: {ckpt["val_accuracy"]:.4f}')

---
## 1️⃣1️⃣ Interview Questions — Demonstrated in Code

In [ ]:
# Q1: Why call optimizer.zero_grad()? What happens if we don't?
import torch
import torch.nn as nn

model_q1 = nn.Linear(3, 1)
optimizer_q1 = torch.optim.SGD(model_q1.parameters(), lr=0.1)
x = torch.tensor([[1.0, 2.0, 3.0]])
y = torch.tensor([[1.0]])

print('WITHOUT zero_grad — gradients ACCUMULATE (wrong!):')
for step in range(3):
    # NO zero_grad!
    out  = model_q1(x)
    loss = nn.MSELoss()(out, y)
    loss.backward()
    grad = model_q1.weight.grad.clone()
    print(f'  Step {step+1} grad: {grad.tolist()}')

# Reset
model_q1 = nn.Linear(3, 1)
optimizer_q1 = torch.optim.SGD(model_q1.parameters(), lr=0.1)

print('\nWITH zero_grad — gradients correct each step:')
for step in range(3):
    optimizer_q1.zero_grad()   # Clear!
    out  = model_q1(x)
    loss = nn.MSELoss()(out, y)
    loss.backward()
    grad = model_q1.weight.grad.clone()
    print(f'  Step {step+1} grad: {grad.tolist()}')

In [ ]:
# Q2: Difference between view() and reshape()
import torch

t = torch.arange(12).float()
print(f'Original: {t.shape}')           # (12,)
print(f'Is contiguous: {t.is_contiguous()}')

v = t.view(3, 4)                         # Works — contiguous
r = t.reshape(3, 4)                      # Works — same as view here
print(f'view(3,4)   : {v.shape}')
print(f'reshape(3,4): {r.shape}')

# After transpose, tensor is NOT contiguous
t2 = torch.randn(3, 4).T               # Transpose → non-contiguous
print(f'\nAfter transpose, is contiguous: {t2.is_contiguous()}')
try:
    v2 = t2.view(-1)
    print('view worked')
except RuntimeError as e:
    print(f'view FAILED: {e}')
r2 = t2.reshape(-1)                     # reshape() handles non-contiguous
print(f'reshape worked: {r2.shape}')

# Fix: make contiguous first, then view
v3 = t2.contiguous().view(-1)
print(f'contiguous().view() worked: {v3.shape}')

In [ ]:
# Q3: model.train() vs model.eval() — see the DIFFERENCE
import torch
import torch.nn as nn

# A model with Dropout
model_q3 = nn.Sequential(nn.Dropout(0.5))
x = torch.ones(1, 10)

# In TRAINING mode: neurons get zeroed randomly
model_q3.train()
out_train = [model_q3(x).tolist() for _ in range(3)]
print('train() — dropout active (different each run):')
for o in out_train:
    print(' ', o)

# In EVAL mode: all neurons active (scaled)
model_q3.eval()
out_eval = [model_q3(x).tolist() for _ in range(3)]
print('\neval() — dropout disabled (same every run):')
for o in out_eval:
    print(' ', o)

print('\nKey rule: ALWAYS call model.eval() before inference/testing!')

---
## ✅ PyTorch Training Loop Cheatsheet

```python
# ── Setup ─────────────────────────────────────────────────
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = MyModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ── Training ──────────────────────────────────────────────
for epoch in range(num_epochs):
    model.train()                          # Enable dropout, BN train mode
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()              # 1. Clear gradients
        output = model(X_batch)            # 2. Forward pass
        loss   = criterion(output, y_batch)# 3. Compute loss
        loss.backward()                    # 4. Backprop
        optimizer.step()                   # 5. Update weights

    model.eval()                           # Disable dropout
    with torch.no_grad():                  # No gradient tracking
        for X_val, y_val in val_loader:
            pred = model(X_val.to(device))

# ── Save/Load ─────────────────────────────────────────────
torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))
```

---
*Last updated: May 2026*